# EXAMPLE (staff only): Mission 1 filled in with very simple code

This copy shows what a completed notebook looks like. The ideas are deliberately basic:
- **Selection:** pick the 100 longest entries.
- **Training:** 1 epoch at a default learning rate.
- **Benchmark:** perplexity on a few hand-written entries in Tinystories Instruct format (negated, so higher is better).

Do not release this file to students.

### 1. Setup

Do these setup steps each time you open the notebook:
1. In Colab, choose **Runtime → Change runtime type → T4 GPU**.
2. **Upload `common.py`.** Open the file browser on the left and upload it. Colab deletes uploaded files when the session ends, so upload it again each session.
3. **Run the cell below.**

`common.py` contains the shared code that staff also use for grading. Do not edit it.

In [1]:
!rm -f common.py
!wget -q https://raw.githubusercontent.com/sadhikamalladi/cs151b-fa26/project_testing/common.py
!head -3 common.py

"""CSE 151B Fall 2026 course project starter code:

Every mission notebook imports this file. Do not modify it: staff use the same


In [2]:
!pip install -q transformers datasets

import os
assert os.path.exists("common.py"), "Put common.py in the same folder as this notebook."
import common

common.set_seed()
os.makedirs("submission/benchmark_data", exist_ok=True)
print("Device:", common.get_device())

Device: cuda


### 2. Load data and the base model

In [ ]:
pool, val = common.load_instruct()

model, tokenizer = common.load_model()
print(f"Base model validation perplexity: {common.compute_perplexity(model, tokenizer, val):.2f}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

Loaded 10,000 training stories and 1,000 validation stories.


config.json:   0%|          | 0.00/968 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/722 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  291MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

KeyboardInterrupt: 

**What an entry looks like:** 

Each entry comes from [TinyStoriesInstruct](https://huggingface.co/datasets/roneneldan/TinyStoriesInstruct), where each story was written to match a set of instructions. The instructions appear as header lines before the story (such as `Summary:`, `Words:`, and `Features:`), followed by `Story:` and the story itself. 

Run the cell below to see a few entries and how they differ. How you use the headers and the story in training is up to you. Scoring always uses perplexity on full entries, in the format shown.

In [ ]:
for entry in pool[:3]:
    print(entry, "\n" + "-" * 60)

### 3. Your algorithm

`select_examples` returns the indices of the entries to train on. `train` fine-tunes the model on them. Staff score the model you save in section 5, and may rerun this code to check that it follows the rules and reproduces your model.

`common.finetune()` accepts `epochs`, `max_steps`, `lr`, and `batch_size`, or you can write your own training loop. Run `help(common.finetune)` in a cell to see all options.

In [ ]:
%%writefile submission/algorithm.py
import common


def select_examples(pool: list[str], k: int) -> list[int]:
    """Return the indices of at most k distinct entries in `pool` to train on."""
    # longer entries contain more text to learn from, so pick the k longest.
    by_length = sorted(range(len(pool)), key=lambda i: len(pool[i].split()), reverse=True)
    return by_length[:k]


def train(model, tokenizer, examples: list[str]):
    """Fine-tune `model` on `examples` and return it."""
    # one pass with the default settings, on the full entries.
    common.finetune(model, tokenizer, examples, epochs=1)
    return model


Writing submission/algorithm.py


In [ ]:
select_examples = common.load_function("submission/algorithm.py", "select_examples")
train = common.load_function("submission/algorithm.py", "train")

idx = select_examples(pool, common.M1_MAX_EXAMPLES)
assert len(idx) <= common.M1_MAX_EXAMPLES, f"Selected {len(idx)} examples; the limit is {common.M1_MAX_EXAMPLES}."
assert len(set(idx)) == len(idx), "Indices must be distinct."
assert all(0 <= i < len(pool) for i in idx), "Indices must refer to entries in the pool."

model, tokenizer = common.load_model()
model = train(model, tokenizer, [pool[i] for i in idx])
ppl = common.compute_perplexity(model, tokenizer, val)
print(f"Your validation perplexity: {ppl:.2f}")

Loading weights:   0%|          | 0/56 [00:00<?, ?it/s]

Finished: 13 steps in 5.6s, final loss 0.8658
Your validation perplexity: 3.54


### 4. Your team benchmark

Requirements:
- `team_benchmark(model, tokenizer)` returns **one number, where higher is better**
- it returns the same number every time it runs (seed any randomness)
- it finishes within `common.BENCHMARK_TIME_LIMIT_S` seconds and does not change the model's weights
- any data it needs goes in `submission/benchmark_data/`, and none of it may come from the training data loaded above

In [ ]:
%%writefile submission/benchmark.py
"""Team benchmark."""
import common

# Hand-written entries in the TinyStoriesInstruct format (not from the training pool).
ENTRIES = [
    "Summary: Mia opens a box from Dad and finds a red kite.\nWords: box, kite, surprise\nFeatures: Dialogue\nStory:\n"
    'Mia looked at the box. "What is inside?" she asked. "A surprise," said Dad. Mia opened it and found a red kite.',
    "Summary: Ben asks to go to the park and Mom says yes after lunch.\nFeatures: Dialogue\nStory:\n"
    '"Can we go to the park?" asked Ben. Mom smiled. "Yes, after lunch," she said. Ben ate fast and put on his shoes.',
    "Words: frog, swim, rock\nSummary: A duck invites a frog to swim, and they swim together.\nStory:\n"
    'The frog sat on a rock. "Hello," said the duck. "Do you want to swim?" The frog jumped in, and they swam together.',
    "Features: Dialogue, Twist\nSummary: Sam loses his hat and his sister finds it under the bed.\nStory:\n"
    '"I lost my hat!" cried Sam. His sister helped him look. "Here it is, under the bed," she said. Sam laughed and hugged her.',
]


def team_benchmark(model, tokenizer) -> float:
    """Evaluate `model` and return one number. Higher must mean better."""
    # Perplexity is lower-is-better, so return its negative to make higher better.
    return -common.compute_perplexity(model, tokenizer, ENTRIES)

Writing submission/benchmark.py


In [ ]:
team_benchmark = common.load_function("submission/benchmark.py", "team_benchmark")

base_model, base_tokenizer = common.load_model()
print("Base model:")
common.check_team_benchmark(team_benchmark, base_model, base_tokenizer)

print("Your model:", team_benchmark(model, tokenizer))

Loading weights:   0%|          | 0/56 [00:00<?, ?it/s]

Base model:
PASS: score -7.2304, runtime 0.0s (limit 300s)
Your model: -7.557578096423202


### 5. Save your submission

Staff load your model with `common.load_model("submission/model")`, so check that it reloads with the same validation perplexity before submitting (use the cell below to check). If you used LoRA, call `model = model.merge_and_unload()` before saving. If you pruned with `torch.nn.utils.prune`, call `prune.remove(...)` on each pruned layer first.

Run this when you are ready to submit. Then upload the zip file to Gradescope.

Your `submission/` folder should contain:
- `algorithm.py`
- `benchmark.py` and `benchmark_data/`
- `model/`

In [ ]:
common.save_model(model, tokenizer)

# Check that the saved model reloads correctly (should match your validation perplexity above)
reloaded, _ = common.load_model("submission/model")
print(f"Reloaded model validation perplexity: {common.compute_perplexity(reloaded, tokenizer, val):.2f}")

common.package_submission("mission1")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model to submission/model (278 MB).


Loading weights:   0%|          | 0/56 [00:00<?, ?it/s]

Reloaded model validation perplexity: 3.54
Created /content/mission1_submission.zip (255 MB). Upload this file to Gradescope.


'/content/mission1_submission.zip'

In [ ]:
!unzip -l mission1_submission.zip

Archive:  mission1_submission.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
        0  2026-10-28 17:49   submission/
        0  2026-10-28 17:49   submission/__pycache__/
        0  2026-10-28 17:48   submission/benchmark_data/
        0  2026-10-28 17:49   submission/model/
      616  2026-10-28 17:49   submission/algorithm.py
      881  2026-10-28 17:49   submission/benchmark.py
274062040  2026-10-28 17:49   submission/model/model.safetensors
     1000  2026-10-28 17:49   submission/model/config.json
  3557945  2026-10-28 17:49   submission/model/tokenizer.json
      140  2026-10-28 17:49   submission/model/generation_config.json
      326  2026-10-28 17:49   submission/model/tokenizer_config.json
     1114  2026-10-28 17:49   submission/__pycache__/algorithm.cpython-313.pyc
     1001  2026-10-28 17:49   submission/__pycache__/benchmark.cpython-313.pyc
---------                     -------
277625063                     13 files
